# Stress Prediction v20 — Faithful v16 Reproduction (Safe Baseline)

This is a clean, faithful reproduction of the v16 submission that scored **0.377** on public LB. Every change we tried over v16 (v17=0.315, v19=0.311) made things worse. We're returning to the proven recipe to lock in the safe baseline before trying any new direction.

## v16 recipe (preserved exactly)
- Conservative cleaning, no row deletion
- v7c feature extractor: per-window stats over 3-minute lookback (mean, std, min/max, median, skew, kurt, IQR, slope, delta, t1/t3 means)
- HRV time-domain features (sdnn, rmssd, pnn25/50, mean_rr, cv_rr)
- Accelerometer magnitude features
- **`pid_enc` included** as feature (despite v17 theory it was leakage — empirically it helped)
- LightGBM with `class_weight='balanced'` + manual `sample_weight`
- **StratifiedKFold 5-fold × 7 seeds** ensemble (NOT GroupKFold — that's reporting-only)
- Median imputation
- **`proba × train_prior^1.6`** calibration
- **Session smoothing** at strength 0.30 (within-session probability averaging)

Output: `submission.csv`. Expected LB: ~0.377.


In [11]:
%pip -q install lightgbm scikit-learn pandas numpy scipy


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats as spstats

from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold

import lightgbm as lgb

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path('.')
TRAIN_DATA  = pd.read_csv(DATA_DIR / 'train-sensor.csv')
TRAIN_LABEL = pd.read_csv(DATA_DIR / 'train-label.csv')
TEST_DATA   = pd.read_csv(DATA_DIR / 'test-sensor.csv')
TEST_LABEL  = pd.read_csv(DATA_DIR / 'test-label.csv')

print('Raw shapes')
print('  TRAIN_DATA :', TRAIN_DATA.shape)
print('  TRAIN_LABEL:', TRAIN_LABEL.shape)
print('  TEST_DATA  :', TEST_DATA.shape)
print('  TEST_LABEL :', TEST_LABEL.shape)


Raw shapes
  TRAIN_DATA : (4694400, 8)
  TRAIN_LABEL: (815, 4)
  TEST_DATA  : (5921280, 8)
  TEST_LABEL : (1028, 4)


## Cleaning (v16 conservative)

In [13]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

def clean_sensor(df):
    out = df.copy()
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    for c in SENSOR_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(float)
    out['accel_x'] = out['accel_x'].clip(-128, 127)
    out['accel_y'] = out['accel_y'].clip(-128, 127)
    out['accel_z'] = out['accel_z'].clip(-128, 127)
    out['eda'] = out['eda'].clip(0, 60)
    out['heart_rate'] = out['heart_rate'].clip(40, 190)
    out['temperature'] = out['temperature'].clip(20, 40)
    return out.sort_values(['pid', 'timestamp']).reset_index(drop=True)

def clean_label(df):
    out = df.copy()
    out['id'] = pd.to_numeric(out['id'], errors='raise').astype(int)
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    out['stress'] = pd.to_numeric(out['stress'], errors='coerce')
    return out

TRAIN_DATA = clean_sensor(TRAIN_DATA)
TEST_DATA = clean_sensor(TEST_DATA)
TRAIN_LABEL = clean_label(TRAIN_LABEL)
TEST_LABEL = clean_label(TEST_LABEL)

assert TRAIN_DATA[SENSOR_COLS + ['timestamp']].isna().sum().sum() == 0
assert TEST_DATA[SENSOR_COLS + ['timestamp']].isna().sum().sum() == 0
print('Cleaned, no nulls.')


Cleaned, no nulls.


## v7c Feature Extractor (v16 features, including pid_enc)

In [14]:
WINDOW_MS = 180_000
HALF_MS = 90_000
THIRD_MS = 60_000

def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']:
            f['hrv_' + k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn'] = float(np.std(rr))
    f['hrv_rmssd'] = float(np.sqrt(np.mean(rr_diff ** 2))) if len(rr_diff) else 0.0
    f['hrv_pnn25'] = float(np.mean(np.abs(rr_diff) > 25)) * 100 if len(rr_diff) else 0.0
    f['hrv_pnn50'] = float(np.mean(np.abs(rr_diff) > 50)) * 100 if len(rr_diff) else 0.0
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr'] = f['hrv_sdnn'] / f['hrv_mean_rr'] if f['hrv_mean_rr'] > 1e-6 else 0.0
    return f

def extract_features(label_df, sensor_df, pid_enc_map):
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    rows = []
    for n, lrow in enumerate(label_df.itertuples(index=False), 1):
        pid = lrow.pid; ts = float(lrow.timestamp); lid = int(lrow.id)
        feat = {'id': lid}
        sg = sensor_by_pid.get(pid)
        if sg is None:
            rows.append(feat); continue
        ta = sg['timestamp'].values
        wa  = sg.loc[(ta >= ts - WINDOW_MS) & (ta <= ts), SENSOR_COLS]
        wf  = sg.loc[(ta >= ts - WINDOW_MS) & (ta < ts - HALF_MS), SENSOR_COLS]
        wl  = sg.loc[(ta >= ts - HALF_MS) & (ta <= ts), SENSOR_COLS]
        wt1 = sg.loc[(ta >= ts - WINDOW_MS) & (ta < ts - 2 * THIRD_MS), SENSOR_COLS]
        wt3 = sg.loc[(ta >= ts - THIRD_MS) & (ta <= ts), SENSOR_COLS]
        
        feat['window_count'] = len(wa)
        for c in SENSOR_COLS:
            v = wa[c].dropna().values.astype(float)
            vf = wf[c].dropna().values.astype(float)
            vl = wl[c].dropna().values.astype(float)
            vt1 = wt1[c].dropna().values.astype(float)
            vt3 = wt3[c].dropna().values.astype(float)
            if len(v) == 0:
                for s in ['mean','std','min','max','median','skew','kurt','range','q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[f'{c}_{s}'] = np.nan
                continue
            feat[f'{c}_mean'] = float(np.mean(v))
            feat[f'{c}_std'] = float(np.std(v))
            feat[f'{c}_min'] = float(np.min(v))
            feat[f'{c}_max'] = float(np.max(v))
            feat[f'{c}_median'] = float(np.median(v))
            feat[f'{c}_skew'] = float(spstats.skew(v)) if len(v) > 2 else 0.0
            feat[f'{c}_kurt'] = float(spstats.kurtosis(v)) if len(v) > 2 else 0.0
            feat[f'{c}_range'] = float(np.max(v) - np.min(v))
            feat[f'{c}_q25'] = float(np.percentile(v, 25))
            feat[f'{c}_q75'] = float(np.percentile(v, 75))
            feat[f'{c}_iqr'] = feat[f'{c}_q75'] - feat[f'{c}_q25']
            feat[f'{c}_delta'] = float(np.mean(vl) - np.mean(vf)) if len(vf) and len(vl) else 0.0
            feat[f'{c}_slope'] = float(np.polyfit(np.linspace(0, 1, len(v)), v, 1)[0]) if len(v) > 2 else 0.0
            feat[f'{c}_t1_mean'] = float(np.mean(vt1)) if len(vt1) else float(np.mean(v))
            feat[f'{c}_t3_mean'] = float(np.mean(vt3)) if len(vt3) else float(np.mean(v))
            feat[f'{c}_t3t1'] = feat[f'{c}_t3_mean'] - feat[f'{c}_t1_mean']
        
        ax, ay, az = wa['accel_x'].values, wa['accel_y'].values, wa['accel_z'].values
        if len(ax):
            mag = np.sqrt(ax**2 + ay**2 + az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean'] = feat['accel_mag_std'] = feat['accel_mag_max'] = np.nan
        feat.update(hrv_time_domain(wa['heart_rate']))
        feat['pid_enc'] = pid_enc_map.get(pid, -1)  # KEY: pid_enc kept (v16 used it)
        rows.append(feat)
        if n % 200 == 0:
            print(f'  {n}/{len(label_df)}')
    return pd.DataFrame(rows).set_index('id')

train_pid_map = {p: i for i, p in enumerate(TRAIN_LABEL['pid'].unique())}
print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_map)
print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA, train_pid_map)
print('train:', train_features.shape, 'test:', test_features.shape)


Extracting train features...
  200/815
  400/815
  600/815
  800/815
Extracting test features...
  200/1028
  400/1028
  600/1028
  800/1028
  1000/1028
train: (815, 107) test: (1028, 107)


In [15]:
tli = TRAIN_LABEL.set_index('id')
y = tli.loc[train_features.index, 'stress'].astype(int)
groups = tli.loc[train_features.index, 'pid']

imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(train_features), columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features), columns=test_features.columns, index=test_features.index)

counts = Counter(y)
total = len(y)
class_weights = {0: total / (3 * counts[0]),
                 1: min(total / (3 * counts[1]), 2.5),
                 2: total / (3 * counts[2])}
sample_weights = np.array([class_weights[int(yi)] for yi in y])
train_prior = np.array([counts[i] / total for i in range(3)])

print('X_imp:', X_imp.shape)
print('Class weights:', {k: round(v, 3) for k, v in class_weights.items()})
print('Train prior:', {i: round(train_prior[i], 3) for i in range(3)})


X_imp: (815, 107)
Class weights: {0: 1.677, 1: 2.5, 2: 0.463}
Train prior: {0: np.float64(0.199), 1: np.float64(0.081), 2: np.float64(0.72)}


## Session Helpers (v16 session smoothing)

In [16]:
def make_session_groups(label_df, gap_ms=30 * 60 * 1000):
    labels = label_df.copy().reset_index(drop=True)
    labels['rowpos'] = np.arange(len(labels))
    groups_out = []
    for pid, grp in labels.sort_values(['pid', 'timestamp']).groupby('pid', sort=False):
        ts = grp['timestamp'].values.astype(float)
        sess = np.cumsum(np.r_[0, np.diff(ts) > gap_ms])
        for sid in np.unique(sess):
            groups_out.append(grp['rowpos'].values[sess == sid])
    return groups_out

train_label_for_rows = TRAIN_LABEL.set_index('id').loc[X_imp.index].reset_index()
TRAIN_SESSIONS = make_session_groups(train_label_for_rows)
TEST_SESSIONS = make_session_groups(TEST_LABEL)

def smooth_by_session(proba, sessions, strength=0.30):
    out = proba.copy()
    for idx in sessions:
        mean = proba[idx].mean(axis=0, keepdims=True)
        out[idx] = (1 - strength) * proba[idx] + strength * mean
    return out

print('Train sessions:', len(TRAIN_SESSIONS), 'Test sessions:', len(TEST_SESSIONS))


Train sessions: 67 Test sessions: 106


## v16 Final Model — 7 Seeds × 5-Fold StratifiedKFold

NOTE: This CV is leaky (same subjects in train and val), so reported CV BA is **not** a realistic LB estimate. The CV BA here will look high (~0.80) but actual LB will be ~0.377. This is the v16 recipe; we know empirically it works.

In [17]:
LGBM_PARAMS = dict(
    n_estimators=1000,
    learning_rate=0.02,
    num_leaves=127,
    max_depth=-1,
    min_child_samples=5,
    subsample=0.6,
    colsample_bytree=0.6,
    reg_alpha=0.3,
    reg_lambda=0.3,
    class_weight='balanced',
    objective='multiclass',
    num_class=3,
    n_jobs=-1,
    verbose=-1,
)

SEEDS = [42, 7, 123, 17, 99, 256, 314]  # 7 seeds, same as v16
N_SPLITS = 5
all_test_proba = []
all_cv_scores = []

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_proba = np.zeros((len(X_test_imp), 3))
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y), 1):
        model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        model.fit(
            X_imp.iloc[tr_idx], y.iloc[tr_idx],
            sample_weight=sample_weights[tr_idx],
            eval_set=[(X_imp.iloc[val_idx], y.iloc[val_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)],
        )
        val_pred = model.predict(X_imp.iloc[val_idx])
        score = balanced_accuracy_score(y.iloc[val_idx], val_pred)
        fold_scores.append(score)
        seed_proba += model.predict_proba(X_test_imp)
    seed_proba /= N_SPLITS
    all_test_proba.append(seed_proba)
    all_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed} mean (leaky) CV={np.mean(fold_scores):.4f}')

raw_test_proba = np.mean(all_test_proba, axis=0)
print(f'\nEnsemble (leaky) stratified CV mean = {np.mean(all_cv_scores):.4f}')
print('Note: this CV is artificially high due to subject leakage. Real LB will be ~0.377.')
print('Raw test argmax distribution:', dict(Counter(np.argmax(raw_test_proba, axis=1))))


  Seed 42 mean (leaky) CV=0.8121
  Seed 7 mean (leaky) CV=0.8126
  Seed 123 mean (leaky) CV=0.7910
  Seed 17 mean (leaky) CV=0.8044
  Seed 99 mean (leaky) CV=0.8079
  Seed 256 mean (leaky) CV=0.8062
  Seed 314 mean (leaky) CV=0.7949

Ensemble (leaky) stratified CV mean = 0.8042
Note: this CV is artificially high due to subject leakage. Real LB will be ~0.377.
Raw test argmax distribution: {np.int64(2): 137, np.int64(0): 371, np.int64(1): 520}


## v16 Submission — alpha=1.6, session smoothing 0.30

In [18]:
DEFAULT_ALPHA = 1.6      # v16's proven calibration strength
SMOOTH_STRENGTH = 0.30   # v16's proven session smoothing strength

cal = raw_test_proba * (train_prior ** DEFAULT_ALPHA)
cal = cal / cal.sum(axis=1, keepdims=True)
cal_smooth = smooth_by_session(cal, TEST_SESSIONS, strength=SMOOTH_STRENGTH)
final_preds = np.argmax(cal_smooth, axis=1).astype(int)

submission = pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': final_preds})
submission.to_csv('submission.csv', index=False)

print('Saved submission.csv')
print('alpha:', DEFAULT_ALPHA, '| session smoothing:', SMOOTH_STRENGTH)
print('Final test distribution:', dict(Counter(final_preds)))
print('Train prior            :', train_prior.round(3).tolist())

test_fracs = np.bincount(final_preds, minlength=3) / len(final_preds)
print('Test fractions         :', test_fracs.round(3).tolist())
print('Max abs deviation      :', round(np.abs(test_fracs - train_prior).max(), 3))
print()
print(submission.head(10))
print()
print('=== EXPECTED LB: ~0.377 (faithful v16 reproduction) ===')


Saved submission.csv
alpha: 1.6 | session smoothing: 0.3
Final test distribution: {np.int64(2): 827, np.int64(0): 160, np.int64(1): 41}
Train prior            : [0.199, 0.081, 0.72]
Test fractions         : [0.156, 0.04, 0.804]
Max abs deviation      : 0.084

     id  stress
0  1227       2
1  1228       0
2  1229       2
3  1230       2
4  1231       2
5  1232       2
6  1233       0
7  1234       2
8  1235       0
9  1236       0

=== EXPECTED LB: ~0.377 (faithful v16 reproduction) ===


In [19]:
import pandas as pd
from collections import Counter
s = pd.read_csv('submission.csv')
print('Current submission.csv distribution:', dict(Counter(s['stress'])))
# If it shows {2: 987, 0: 36, 1: 5} -> it's smooth_1.0 (the bad one)
# If it shows {2: 827, 0: 160, 1: 41} -> it's smooth_0.30 (the good one)

# Force-overwrite with the safe v16-faithful version:
import shutil
shutil.copy('submission_smooth_0.30.csv', 'submission.csv')
s = pd.read_csv('submission.csv')
print('After fix:', dict(Counter(s['stress'])))

Current submission.csv distribution: {2: 827, 0: 160, 1: 41}
After fix: {2: 827, 0: 160, 1: 41}
